In [1]:
import jax
import jax.numpy as jnp
import optax
from tqdm import tqdm
import numpy as np
import json
import time

import pennylane as qml

In [2]:
import os, sys
sys.path.append('../')

from pqcqec.utils.constants import QUBITS_FOR_GATES, QISKIT_GATES, GATE_IS_DIRECTIONAL, PENNYLANE_GATES
from pqcqec.noise.simple_noise import PennylaneNoisyGates
# from pqcqec.simulate.simulate import run_circuit_with_noise_model
from pqcqec.circuits.modify import pennylane_state_embedding

In [3]:
PQC_GATES = ['rz', 'rx', 'rz']
DATA_PATH = '../nogit/circuit_tokens/no_uncomp/5q_500g_circuit_data/'
GOOD_DATA_PATH = DATA_PATH + 'per_seed_data/'
BAD_DATA_PATH = DATA_PATH + 'poor_fidelity/'
CONFIG_PATH = DATA_PATH + 'config.json'

with open(CONFIG_PATH, 'r') as f:
    CONFIG = json.load(f)


NUM_QUBITS = CONFIG.get("qubits", 3)[0]
NUM_GATES = CONFIG.get("gates", 4)[0] # Multiply by 2 for uncomp gates. 
GATE_BLOCKS = CONFIG.get("gate_blocks", 4)
VALID_GATES = CONFIG.get("gate_dist", QISKIT_GATES)
if VALID_GATES:
    VALID_GATES = list(VALID_GATES.keys())
else:
    VALID_GATES = ['x', 'z', 'h', 'cx', 'cz']

print(f"Using {NUM_QUBITS} qubits, {NUM_GATES} gates, {GATE_BLOCKS} gate blocks, {VALID_GATES} valid gates.")

NOISE_DIST = {"x_rad": 0.01, "z_rad": 0.01, "delta_x": 0, "delta_z": 0}

PAD_ID = 0
UNDIRECTED_GATES = [gate for gate in VALID_GATES if not GATE_IS_DIRECTIONAL.get(gate, False)]
print(UNDIRECTED_GATES)

TRAIN_SZ = 0.8
VAL_SZ = 0.1
TEST_SZ = 0.1
BATCH_SIZE = 64

LEARNING_RATE = 5e-6
WEIGHT_DECAY = 1e-3


Using 5 qubits, 500 gates, 4 gate blocks, ['x', 'z', 'h', 'cx', 'cz'] valid gates.
['x', 'z', 'h', 'cz']


In [4]:

good_data = []
poor_data = []

for i, filename in enumerate(os.listdir(GOOD_DATA_PATH)):
    if i > 1000:
        break
    with open(GOOD_DATA_PATH + filename, 'r') as f:
        token_dict = json.load(f)
        good_data.append(token_dict['base_circuit_tokens'])
        f.close()

# for filename in os.listdir(BAD_DATA_PATH):
#     with open(BAD_DATA_PATH + filename, 'r') as f:
#         token_dict = json.load(f)
#         poor_data.append((token_dict['base_circuit_tokens'], token_dict['pqc_params'], token_dict['fidelity']))
#         f.close()

print(f"Number of good data samples: {len(good_data)}")
# print(f"Number of poor data samples: {len(poor_data)}")

Number of good data samples: 1001


In [ ]:

def simple_circuit_simulator(circuit_ops, input_state, num_qubits, x_noise, z_noise):
    """Runs a quantum circuit with a noise model using PennyLane and PyTorch."""
    qdevice = qml.device("default.qubit", wires=num_qubits)

    qml.qjit
    @qml.qnode(qdevice)
    def circuit(state):
        pennylane_state_embedding(state, num_qubits)
        for i, op in enumerate(circuit_ops):
            gate, wires, param = op
            # noise_model.apply_gate(gate, wires, angle=param)
            PENNYLANE_GATES[gate](wires=wires)
            for wire in wires:
                qml.RX(x_noise[i], wires=[wire])  # Example noise application
                qml.RZ(z_noise[i], wires=[wire])  # Example noise application

        return qml.state()

    # The function now directly returns the output of the torch interface qnode
    return circuit(input_state)


def simple_circuit_generator(circuit_ops, num_qubits, x_noise, z_noise):
    """Runs a quantum circuit with a noise model using PennyLane and PyTorch."""
    qdevice = qml.device("default.qubit", wires=num_qubits)

    @qml.qjit
    @qml.qnode(qdevice)
    def circuit(state):
        pennylane_state_embedding(state, num_qubits)
        for i, op in enumerate(circuit_ops):
            gate, wires, param = op
            # noise_model.apply_gate(gate, wires, angle=param)
            PENNYLANE_GATES[gate](wires=wires)
            for wire in wires:
                qml.RX(x_noise[i], wires=[wire])  # Example noise application
                qml.RZ(z_noise[i], wires=[wire])  # Example noise application

        return qml.state()

    # The function now directly returns the interface qnode
    return circuit

In [6]:
input_states = np.zeros((100, 2**NUM_QUBITS))
input_states[:,0] = 1.0
x_noise = np.ones(NUM_GATES) * 0.01
z_noise = np.ones(NUM_GATES) * 0.01


In [7]:

# for data in tqdm(good_data):
#     output_state = simple_circuit_simulator(data, input_states, NUM_QUBITS, x_noise, z_noise)
#     # print(f"Output state: {output_state}")

In [8]:

start_time = time.time()
output_state = simple_circuit_simulator(good_data[0], input_states, NUM_QUBITS, x_noise, z_noise)
end_time = time.time()

execution_time = end_time - start_time
print(f"Execution time: {execution_time} seconds")
print(f"Output state: {output_state.shape}")


Execution time: 0.1145787239074707 seconds
Output state: (100, 32)


In [9]:

start_time = time.time()
output_state = simple_circuit_simulator(good_data[10], input_states[0], NUM_QUBITS, x_noise, z_noise)
end_time = time.time()

execution_time = end_time - start_time
print(f"Execution time: {execution_time} seconds")
print(f"Output state: {output_state.shape}")


Execution time: 0.05252575874328613 seconds
Output state: (32,)


In [10]:
circuit_nodes = []
start_time = time.time()
for data in tqdm(good_data):
    # input_state = np.zeros((100, 2**NUM_QUBITS))
    # input_state[:,0] = 1.0
    exec_node = simple_circuit_generator(data, NUM_QUBITS, x_noise, z_noise)
    circuit_nodes.append(exec_node)
end_time = time.time()
print(f"Execution time - circuit creation: {end_time - start_time} seconds")
    # print(f"Output state: {output_state}")

100%|██████████| 1001/1001 [00:00<00:00, 32781.05it/s]

Execution time - circuit creation: 0.03893685340881348 seconds


In [11]:
print(f'Running circuit exec nodes with 1 state to set up compiled nodes...')
for exec_node in tqdm(circuit_nodes):
    output_state = exec_node(input_states[0])
    # print(f"Output state: {output_state.shape}")

Running circuit exec nodes with 1 state to set up compiled nodes...


100%|██████████| 1001/1001 [00:58<00:00, 16.97it/s]


In [12]:
print(f'Running circuit exec nodes with all 100 states...')

for exec_node in tqdm(circuit_nodes):
    output_state = exec_node(input_states)
    # print(f"Output state: {output_state.shape}")


Running circuit exec nodes with all 100 states...


100%|██████████| 1001/1001 [01:54<00:00,  8.76it/s]


In [13]:
print(f'Running circuit exec nodes again...')


for exec_node in tqdm(circuit_nodes):
    output_state = exec_node(input_states)
    # print(f"Output state: {output_state.shape}")



Running circuit exec nodes again...


100%|██████████| 1001/1001 [01:54<00:00,  8.77it/s]


In [14]:
start_time = time.time()
circuit_nodes[0](input_states)
end_time = time.time()

execution_time = end_time - start_time
print(f"Execution time - 100 states: {execution_time} seconds")


Execution time - 100 states: 0.1426098346710205 seconds


In [15]:

start_time = time.time()
circuit_nodes[0](input_states[0])
end_time = time.time()

execution_time = end_time - start_time
print(f"Execution time - 1 states: {execution_time} seconds")


Execution time - 1 states: 0.05745410919189453 seconds


In [16]:
print(f'Running circuit exec nodes for just 10 states...')

for exec_node in tqdm(circuit_nodes):
    output_state = exec_node(input_states[:10])
    # print(f"Output state: {output_state.shape}")


Running circuit exec nodes for just 10 states...


100%|██████████| 1001/1001 [01:06<00:00, 15.01it/s]


In [17]:
start_time = time.time()
circuit_nodes[0](input_states[:10])
end_time = time.time()

execution_time = end_time - start_time
print(f"Execution time - 10 states: {execution_time} seconds")


Execution time - 10 states: 0.06330609321594238 seconds
